# CRT Trading Bot - Colab Runner
Mounts Google Drive, installs deps, runs paper trader with catch-up.

In [ ]:
# 1. Mount Google Drive (click the link, paste auth code)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive/MyDrive/crt_trading')

In [ ]:
# 2. Install dependencies
!pip install -q ccxt pandas numpy flask flask-cors pyngrok
print('Dependencies installed')

In [ ]:
# 3. Copy the CRT bot code to Colab
#    IMPORTANT: Upload the inner 'CRT_Trading_Bot' folder (the one with
#    strategies/, paper/, colab/, backtest/ etc.) to:
#    MyDrive/CRT_Trading_Bot/CRT_Trading_Bot/
import os, sys, shutil

DRIVE_PATH = '/content/drive/MyDrive/crt_trading'
os.makedirs(DRIVE_PATH, exist_ok=True)

# Try common upload locations (inner CRT_Trading_Bot folder)
POSSIBLE_SRC = [
    '/content/drive/MyDrive/CRT_Trading_Bot/CRT_Trading_Bot',  # nested
    '/content/drive/MyDrive/CRT_Trading_Bot',                   # flat
]
COPIED = False
for SRC in POSSIBLE_SRC:
    if os.path.exists(SRC) and os.path.isdir(os.path.join(SRC, 'colab')):
        print(f'Found code at: {SRC}')
        # Copy to /content/ so imports work
        for item in ['colab', 'strategies', 'risk', 'data', 'paper', 'backtest', 'config']:
            src_path = os.path.join(SRC, item)
            dst_path = os.path.join('/content/CRT_Trading_Bot', item)
            if os.path.exists(src_path):
                if os.path.exists(dst_path):
                    shutil.rmtree(dst_path)
                shutil.copytree(src_path, dst_path)
        # ---- Auto-patch colab_trader.py to support exchange fallback ----
        patch_path = '/content/CRT_Trading_Bot/colab/colab_trader.py'
        if os.path.exists(patch_path):
            with open(patch_path) as f:
                code = f.read()
            needs_save = False
            # Patch 1: add exchange_id param if missing
            if 'exchange_id' not in code:
                code = code.replace(
                    'ccxt.binance({"enableRateLimit": True})',
                    'getattr(ccxt, self.exchange_id)({"enableRateLimit": True})'
                )
                code = code.replace('Connected to binance', 'Connected to {self.exchange_id}')
                code = code.replace(
                    'drive_path: str = "/content/drive/MyDrive/crt_trading",',
                    'drive_path: str = "/content/drive/MyDrive/crt_trading",\n        exchange_id: str = "kucoin",'
                )
                code = code.replace(
                    'self.drive_path = Path(drive_path)',
                    'self.exchange_id = exchange_id\n        self.drive_path = Path(drive_path)'
                )
                needs_save = True
            # Patch 2: add exchange fallback logic to _init_exchange
            if '_init_exchange(self)' in code and 'fallbacks' not in code:
                old = '''    def _init_exchange(self):
        try:
            import ccxt
            self.exchange = getattr(ccxt, self.exchange_id)({"enableRateLimit": True})
            self.exchange.load_markets()
            log.info(f"Connected to {self.exchange_id}")
        except Exception as e:
            log.error(f"Exchange init failed: {e}")
            raise'''
                new = '''    def _init_exchange(self):
        import ccxt
        fallbacks = [self.exchange_id, 'kucoin', 'bybit', 'binance', 'mexc', 'gate']
        fallbacks = list(dict.fromkeys(fallbacks))
        for exch_id in fallbacks:
            try:
                self.exchange = getattr(ccxt, exch_id)({"enableRateLimit": True})
                self.exchange.load_markets()
                self.exchange_id = exch_id
                log.info(f"Connected to {exch_id}")
                return
            except Exception as e:
                log.warning(f"Exchange {exch_id} failed: {e}")
        raise RuntimeError(f"No exchange available from: {fallbacks}")'''
                code = code.replace(old, new)
                needs_save = True
            if needs_save:
                with open(patch_path, 'w') as f:
                    f.write(code)
                print('Patched colab_trader.py with exchange fallback')
            else:
                print('colab_trader.py already patched')
        COPIED = True
        break

if not COPIED:
    print('Please upload the inner CRT_Trading_Bot folder to your Drive.')
    print('It should contain: colab/, strategies/, paper/, backtest/, etc.')

In [ ]:
# 4. Configuration - EDIT THESE VALUES
EXCHANGE = 'kucoin'             # 'kucoin' (global, no geo-block), 'bybit'/'binance' if accessible
SYMBOL = 'BTC/USDT'
ENTRY_MODE = 'limit_retest'    # 'immediate', 'limit_retest', 'ltf'
USE_LTF = False
INITIAL_CAPITAL = 10000
RISK_PER_TRADE = 0.02
FEE_RATE = 0.0004              # 0.04% avg

DRIVE_PATH = '/content/drive/MyDrive/crt_trading'
print(f'Config: {EXCHANGE} {SYMBOL} mode={ENTRY_MODE} capital=${INITIAL_CAPITAL} fee={FEE_RATE*100:.2f}%')

In [ ]:
# 5. Run the trader (this will run until Colab disconnects)
#    On restart: just run this cell again — it loads state from Drive and catches up
import os, sys
sys.path.insert(0, '/content/CRT_Trading_Bot')

from colab.colab_trader import ColabTrader

trader = ColabTrader(
    drive_path=DRIVE_PATH,
    exchange_id=EXCHANGE,
    symbol=SYMBOL,
    entry_mode=ENTRY_MODE,
    use_ltf=USE_LTF,
    initial_capital=INITIAL_CAPITAL,
    risk_per_trade=RISK_PER_TRADE,
    fee_rate=FEE_RATE,
)

print('Trader started. Press STOP or wait for Colab timeout.')
print(f'Logs: {DRIVE_PATH}/colab_trader.log')
print(f'State: {DRIVE_PATH}/trader_state.json')
trader.start()

In [ ]:
# 6. (Optional) View live dashboard via ngrok tunnel
#    Run this in a SEPARATE cell while the trader is running
#    This requires pyngrok: pip install pyngrok
from pyngrok import ngrok
from flask import Flask, jsonify
from flask_cors import CORS
import json

app = Flask(__name__); CORS(app)

def load_trader_data():
    state_path = f'{DRIVE_PATH}/trader_state.json'
    if os.path.exists(state_path):
        return json.load(open(state_path))
    return {'error': 'Not running'}

@app.route('/')
def home():
    data = load_trader_data()
    return jsonify(data)

public_url = ngrok.connect(5050)
print(f'Dashboard URL: {public_url}')
app.run(host='0.0.0.0', port=5050)

In [ ]:
# 7. Check status (run this in a separate cell while trader is running)
import json
state_path = f'{DRIVE_PATH}/trader_state.json'
if os.path.exists(state_path):
    s = json.load(open(state_path))
    print(f'Capital: \${s.get("equity",0):,.2f}')
    print(f'Positions: {len(s.get("positions",[]))}')
    print(f'Trades: {s.get("trades_count",0)}')
    print(f'Last 4H: {s.get("last_4h_ts","-")}')
    print(f'Updated: {s.get("updated_at","-")}')
else:
    print('No state file found')

log_path = f'{DRIVE_PATH}/colab_trader.log'
if os.path.exists(log_path):
    print(f'\n=== Last 20 log lines ===')
    !tail -20 "$log_path"